# Imports

In [ ]:
pip install minisom catboost

In [14]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from minisom import MiniSom
from sklearn.neural_network import BernoulliRBM
import tensorflow as tf
from tensorflow import keras
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

# Load data
Adult Income Dataset: https://www.openml.org/search?type=data&sort=version&status=any&order=asc&exact_name=adult

In [15]:
# Load data
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
X = adult.data
y = adult.target
display(adult['data'].head())

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# Handle categoricals
categorical_cols = X.select_dtypes(include=['category', 'object']).columns
for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Fill missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original data shape:", X_train_scaled.shape)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


Original data shape: (39073, 14)


# 1. Dimension Reduction
Reduce the dimension of the original data.

In [16]:
# Dynamic parameters
n_samples = X_train_scaled.shape[0]
n_features = X_train_scaled.shape[1]
n_classes = len(np.unique(y_train))

# SOM parameters
grid_side = int(np.ceil(np.sqrt(5 * np.sqrt(n_samples))))
n_iterations = grid_side * grid_side * 30

# RBM parameters - compress to ~10% of original features
rbm_components = max(n_features // 10, 8)

# Autoencoder parameters - progressive compression by factor of 2
ae_layer1 = max(n_features // 2, 32)
ae_layer2 = max(ae_layer1 // 2, 16)
ae_layer3 = max(ae_layer2 // 2, 8)
ae_bottleneck = max(ae_layer3 // 2, 4)

print(f"\nDynamic Parameters:")
print(f"  SOM: {grid_side}×{grid_side} grid, {n_iterations} iterations")
print(f"  Autoencoder: {n_features} -> {ae_layer1} -> {ae_layer2} -> {ae_layer3} -> {ae_bottleneck}")
print(f"  RBM: {n_features} -> {rbm_components} components")


Dynamic Parameters:
  SOM: 32×32 grid, 30720 iterations
  Autoencoder: 14 -> 32 -> 16 -> 8 -> 4
  RBM: 14 -> 8 components


> ## a) SOM
Self-Organizing Map

In [17]:
start = time.time()

som = MiniSom(grid_side, grid_side, n_features, sigma=1.0, learning_rate=0.5)
som.random_weights_init(X_train_scaled)
som.train_random(X_train_scaled, n_iterations)
X_train_som = np.array([som.winner(x) for x in X_train_scaled])
X_test_som = np.array([som.winner(x) for x in X_test_scaled])

som_time = time.time() - start
print(f"\nSOM completed in {round(som_time, 2)}s, reduced to {X_train_som.shape[1]} features")


SOM completed in 26.85s, reduced to 2 features


> ## b) RBM
Restricted Boltzmann Machine

In [18]:
start = time.time()

rbm = BernoulliRBM(n_components=rbm_components, learning_rate=0.01, n_iter=50, random_state=42, verbose=0)
X_train_rbm = rbm.fit_transform(X_train_scaled)
X_test_rbm = rbm.transform(X_test_scaled)

rbm_time = time.time() - start
print(f"RBM completed in {round(rbm_time, 2)}s, reduced to {X_train_rbm.shape[1]} features")

RBM completed in 19.76s, reduced to 8 features


> ## c) Autoencoder

In [19]:
start = time.time()

encoder = keras.Sequential([
    keras.layers.Dense(ae_layer1, activation='relu', input_shape=(n_features,)),
    keras.layers.Dense(ae_layer2, activation='relu'),
    keras.layers.Dense(ae_layer3, activation='relu'),
    keras.layers.Dense(ae_bottleneck, activation='relu')
])
decoder = keras.Sequential([
    keras.layers.Dense(ae_layer3, activation='relu', input_shape=(ae_bottleneck,)),
    keras.layers.Dense(ae_layer2, activation='relu'),
    keras.layers.Dense(ae_layer1, activation='relu'),
    keras.layers.Dense(n_features, activation='linear')
])
autoencoder = keras.Sequential([encoder, decoder])
autoencoder.compile(optimizer='adam', loss='mse')

# Dynamic epochs based on dataset size
epochs = min(max(10000 // n_samples, 20), 100)
batch_size = min(max(n_samples // 50, 32), 256)

autoencoder.fit(X_train_scaled, X_train_scaled, epochs=epochs, batch_size=batch_size, verbose=0)

X_train_ae = encoder.predict(X_train_scaled, verbose=0)
X_test_ae = encoder.predict(X_test_scaled, verbose=0)

ae_time = time.time() - start
print(f"Autoencoder completed in {round(ae_time, 2)}s, reduced to {X_train_ae.shape[1]} features")

Autoencoder completed in 15.38s, reduced to 4 features


# 2. Classification
Experiment with three classifier algorithms on the four datasets (original, SOM, RBM, Autoencoder):
1. XGBoost
2. LightGBM
3. CatBoost

In [20]:
# ============ 4. Train classifiers on all datasets ============
datasets = {
    'Original': (X_train_scaled, X_test_scaled),
    'SOM': (X_train_som, X_test_som),
    'Autoencoder': (X_train_ae, X_test_ae),
    'RBM': (X_train_rbm, X_test_rbm)
}

results = []

for dataset_name, (X_tr, X_te) in datasets.items():

    # XGBoost
    start = time.time()
    xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', n_estimators=100)
    xgb.fit(X_tr, y_train)
    xgb_pred = xgb.predict(X_te)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    xgb_time = time.time() - start
    results.append(['XGBoost', dataset_name, xgb_acc, xgb_time])

    # LightGBM
    start = time.time()
    lgb = LGBMClassifier(random_state=42, verbose=-1, n_estimators=100)
    lgb.fit(X_tr, y_train)
    lgb_pred = lgb.predict(X_te)
    lgb_acc = accuracy_score(y_test, lgb_pred)
    lgb_time = time.time() - start
    results.append(['LightGBM', dataset_name, lgb_acc, lgb_time])

    # CatBoost
    start = time.time()
    cat = CatBoostClassifier(random_state=42, verbose=0, iterations=100)
    cat.fit(X_tr, y_train)
    cat_pred = cat.predict(X_te)
    cat_acc = accuracy_score(y_test, cat_pred)
    cat_time = time.time() - start
    results.append(['CatBoost', dataset_name, cat_acc, cat_time])

# Summary
results_df = pd.DataFrame(results, columns=['Classifier', 'Dataset', 'Accuracy', 'Time (s)'])
results_df

,Classifier,Dataset,Accuracy,Time (s)
0,XGBoost,Original,0.877060,0.367629
1,LightGBM,Original,0.878288,0.534398
2,CatBoost,Original,0.879210,0.604765
3,XGBoost,SOM,0.828130,0.207971
4,LightGBM,SOM,0.825980,0.271908
5,CatBoost,SOM,0.827618,0.480601
6,XGBoost,Autoencoder,0.785751,0.234505
7,LightGBM,Autoencoder,0.794759,0.311864
8,CatBoost,Autoencoder,0.794657,0.525616
9,XGBoost,RBM,0.764254,0.285456
